<a href="https://colab.research.google.com/github/Godstouch/GNN-Student-Risk-Prediction-/blob/main/Data_Preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
df = pd.read_excel("/content/datasheet.xlsx")

In [ ]:

import pandas as pd
import numpy as np

from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler


df = pd.read_excel("/content/datasheet.xlsx")

print("Original Shape:", df.shape)



if "Student ID" in df.columns:
    df = df.drop_duplicates(subset="Student ID")
else:
    df = df.drop_duplicates()

print("After removing duplicates:", df.shape)


object_cols = df.select_dtypes(include="object").columns

for col in object_cols:

    df[col] = (
        df[col]
        .astype(str)
        .str.strip()
        .str.title()
    )


transport_col = None

for col in df.columns:
    if "transport" in col.lower():
        transport_col = col
        break

if transport_col:

    df[transport_col] = df[transport_col].replace({

        "Walk":"Walking",
        "Walking ":"Walking",
        "On Foot":"Walking",
        "Foot":"Walking",

    })


child_col = None

for col in df.columns:
    if "child" in col.lower():
        child_col = col
        break

if child_col:

    df[child_col] = df[child_col].replace({

        "Yes":"Yes",

        "No":"No",

        "Sometimes":"Sometimes"

    })



categorical_columns = df.select_dtypes(include="object").columns

numerical_columns = df.select_dtypes(include=np.number).columns

for col in categorical_columns:

    df[col] = df[col].fillna(df[col].mode()[0])

for col in numerical_columns:

    df[col] = df[col].fillna(df[col].median())

print("\nMissing Values")

print(df.isnull().sum())


encoders = {}

for col in categorical_columns:

    encoder = LabelEncoder()

    df[col] = encoder.fit_transform(df[col])

    encoders[col] = encoder

print("\nEncoding Completed")


attendance_cols = [

    c for c in df.columns

    if "attendance" in c.lower()

]

df["Attendance_Rate"] = df[attendance_cols].mean(axis=1) * 100


semester_cols = [

    c for c in df.columns

    if "semester" in c.lower()

]

if len(semester_cols) >= 2:

    academic_score = df[semester_cols].mean(axis=1)

elif len(semester_cols) == 1:

    academic_score = df[semester_cols[0]]

else:

    academic_score = pd.Series([70]*len(df))


# Academic performance score
academic_score = (
    df["Semester 1 average"] +
    df["Semester 2 average"]
) / 2

# Academic trend
grade_change = df["Semester difference"]

# Combined risk score
risk_score = (
    0.6 * df["Attendance_Rate"] +
    0.3 * academic_score +
    0.1 * grade_change
)

# Create 3 groups using percentiles
low_threshold = risk_score.quantile(0.30)
high_threshold = risk_score.quantile(0.70)

df["Dropout_Risk"] = np.where(
    risk_score <= low_threshold,
    "High",
    np.where(
        risk_score >= high_threshold,
        "Low",
        "Medium"
    )
)

# Encode labels
risk_encoder = LabelEncoder()

df["Dropout_Label"] = risk_encoder.fit_transform(
    df["Dropout_Risk"]
)

print("\nDropout Label Counts")
print(df["Dropout_Risk"].value_counts())
print(df["Dropout_Risk"].value_counts())


exclude = [

    "Dropout_Label"

]

numeric_cols = df.select_dtypes(include=np.number).columns

numeric_cols = [

    c for c in numeric_cols

    if c not in exclude

]

scaler = StandardScaler()

df[numeric_cols] = scaler.fit_transform(df[numeric_cols])

print("\nNormalization Complete")



output_file = "/content/Cleaned_GAT_datasheet.xlsx"

df.to_excel(output_file,index=False)

print("\nCleaning Completed Successfully")

print("Saved as:",output_file)

Original Shape: (4000, 34)
After removing duplicates: (4000, 34)

Missing Values
Student ID                               0
School                                   0
Name                                     0
Gender                                   0
Class level                              0
Parental educational level               0
Week1_attendance                         0
Week2_attendance                         0
Week3_attendance                         0
Week4_attendance                         0
Week5_attendance                         0
Week6_attendance                         0
Week7_attendance                         0
Week8_attendance                         0
Week9_attendance                         0
Week10_attendance                        0
Week11_attendance                        0
Week12_attendance                        0
Week13_attendance                        0
Week14_attendance                        0
Semester 1 average                       0
Semester 2 avera

In [ ]:
import pandas as pd
import numpy as np
import re

df = pd.read_excel("/content/Dataset.xlsx")  # match the exact filename you uploaded

print(df.shape)
df.head()

(4000, 34)


,Student ID,School,Name,Gender,Class level,Parental educational level,Week1_attendance,Week2_attendance,Week3_attendance,Week4_attendance,...,Family dropout history,Child labor involvement,Travel time to school,Mode of transport,Teacher relationship quality,Peer relationship quality,Extra-curricular activities,Section,Household income level (standardized),Travel time to school (minutes)
0,1,Shining Star Preparatory,Theophilus Yaw Akyen,Female,4A,SHS,1.0,1.00,0.40,1.0,...,No,No,21 minutes,Walking,Poor,Very Good,Football,Kofi Annan,High,21.0
1,2,Shining Star Preparatory,Theophilus Yaw Akyen,Male,4B,SHS,1.0,1.00,0.25,1.0,...,No,Yes,2 Minutes,Walking,Good,Average,Drawing,Kofi Annan,High,2.0
2,3,Shining Star Preparatory,Theophilus Yaw Akyen,Male,7B3,Tertiary,0.5,0.75,1.00,1.0,...,No,No,4 minutes,Walking,Good,Good,Watch cartoons,K.A. Busia,NaN,4.0
3,4,Weweso MA,Asiama Isaac Ebenezer,Male,5,JHS,0.6,1.00,1.00,1.0,...,No,No,30 minutes,Bus,Average,Good,Drawing,Kofi Annan,High,30.0
4,5,Weweso MA,Theophilus Yaw Akyen,Male,6B,Tertiary,0.0,1.00,1.00,1.0,...,Yes,No,3 minutes,Public transport,Good,Good,Dancing,Kwegyir Aggrey,Average,3.0


In [ ]:
def clean_household_income(df):
    df = df.copy()
    before_dtype = df["Household income level"].dtype
    df["Household income level"] = pd.to_numeric(
        df["Household income level"], errors="coerce"
    )
    n_bad = df["Household income level"].isna().sum()
    if n_bad:
        print(f"[household_income] coerced non-numeric entries -> NaN: {n_bad} row(s) "
              f"(original dtype: {before_dtype})")
    return df

df = clean_household_income(df)
df[["Student ID", "Household income level"]]

[household_income] coerced non-numeric entries -> NaN: 335 row(s) (original dtype: object)


,Student ID,Household income level
0,1,3.0
1,2,3.0
2,3,NaN
3,4,3.0
4,5,2.0
...,...,...
3995,3996,NaN
3996,3997,2.0
3997,3998,NaN
3998,3999,3.0


In [ ]:
def standardize_travel_time(df):
    df = df.copy()

    def parse_time(val):
        if pd.isna(val):
            return np.nan
        s = str(val).strip().lower()
        m = re.match(r"(\d+(?:\.\d+)?)\s*(hour|hr|h)s?\b", s)
        if m:
            return float(m.group(1)) * 60
        m = re.match(r"(\d+(?:\.\d+)?)\s*(minute|min|m)s?\b", s)
        if m:
            return float(m.group(1))
        m = re.match(r"(\d+(?:\.\d+)?)$", s)
        if m:
            return float(m.group(1))
        return np.nan

    parsed = df["Travel time to school"].apply(parse_time)
    n_bad = parsed.isna().sum() - df["Travel time to school"].isna().sum()
    if n_bad:
        print(f"[travel_time] {n_bad} entr(y/ies) could not be parsed, set to NaN "
              f"-- inspect these manually")
    df["Travel time to school (minutes)"] = parsed
    return df

df = standardize_travel_time(df)
df[["Student ID", "Travel time to school", "Travel time to school (minutes)"]]

[travel_time] 8 entr(y/ies) could not be parsed, set to NaN -- inspect these manually


,Student ID,Travel time to school,Travel time to school (minutes)
0,1,21 minutes,21.0
1,2,2 Minutes,2.0
2,3,4 minutes,4.0
3,4,30 minutes,30.0
4,5,3 minutes,3.0
...,...,...,...
3995,3996,30 minutes,30.0
3996,3997,3min,3.0
3997,3998,30 minutes,30.0
3998,3999,3 Minutes,3.0


In [ ]:
ATTENDANCE_COLS = [f"Week{i}_attendance" for i in range(1, 15)]

def fix_attendance_dtypes(df):
    df = df.copy()
    for col in ATTENDANCE_COLS:
        if col in df.columns:
            df[col] = df[col].astype(float)
    return df

df = fix_attendance_dtypes(df)
df[ATTENDANCE_COLS].dtypes

,0
Week1_attendance,float64
Week2_attendance,float64
Week3_attendance,float64
Week4_attendance,float64
Week5_attendance,float64
Week6_attendance,float64
Week7_attendance,float64
Week8_attendance,float64
Week9_attendance,float64
Week10_attendance,float64


In [ ]:
def fix_semester_scale(df, target_scale=100):
    df = df.copy()
    for col in ["Semester 1 average", "Semester 2 average"]:
        if col not in df.columns:
            continue
        candidate_mask = df[col] <= 1.5
        rescaled = df[col] * target_scale
        would_overshoot = candidate_mask & (rescaled > target_scale)
        safe_to_fix = candidate_mask & ~would_overshoot

        n_fixed = safe_to_fix.sum()
        n_nan = would_overshoot.sum()
        if n_fixed:
            print(f"[semester_scale] {col}: rescaling {n_fixed} row(s) "
                  f"0-1 scale -> 0-100")
        if n_nan:
            print(f"[semester_scale] {col}: {n_nan} row(s) set to NaN "
                  f"(x100 would exceed {target_scale}, no reliable fix -- "
                  f"will need imputation)")

        df.loc[safe_to_fix, col] = df.loc[safe_to_fix, col] * target_scale
        df.loc[would_overshoot, col] = np.nan

    if {"Semester 1 average", "Semester 2 average"}.issubset(df.columns):
        df["Semester difference"] = df["Semester 2 average"] - df["Semester 1 average"]
    return df

df = fix_semester_scale(df)
df[["Student ID", "Semester 1 average", "Semester 2 average", "Semester difference"]]

[semester_scale] Semester 1 average: rescaling 4 row(s) 0-1 scale -> 0-100
[semester_scale] Semester 2 average: rescaling 129 row(s) 0-1 scale -> 0-100


,Student ID,Semester 1 average,Semester 2 average,Semester difference
0,1,55.85,93.00,37.15
1,2,51.48,72.95,21.47
2,3,52.78,70.55,17.77
3,4,48.68,5.45,-43.23
4,5,52.38,82.81,30.43
...,...,...,...,...
3995,3996,46.29,5.25,-41.04
3996,3997,57.86,64.25,6.39
3997,3998,62.58,65.92,3.34
3998,3999,52.55,2.38,-50.17


In [ ]:
flagged = df[df["Semester difference"].isna() & (df["Semester 2 average"].isna())]
print(f"Total flagged rows: {len(flagged)}")

# Cross-check against attendance: genuinely at-risk students should also
# show low attendance. If a flagged row has HIGH attendance, that's a
# strong signal the low score is a data-entry artifact, not a real outcome.
avg_attendance = df[ATTENDANCE_COLS].mean(axis=1)
flagged_check = df.loc[flagged.index, ["Student ID", "Semester 1 average"]].copy()
flagged_check["avg_attendance"] = avg_attendance.loc[flagged.index]
flagged_check.sort_values("avg_attendance", ascending=False)

Total flagged rows: 65


,Student ID,Semester 1 average,avg_attendance
2798,2799,52.72,1.000000
2420,2421,64.74,1.000000
2271,2272,48.10,0.967857
3570,3571,52.05,0.957143
217,218,52.07,0.953571
...,...,...,...
2890,2891,51.82,0.764286
3961,3962,51.65,0.760714
2669,2670,51.96,0.732143
1029,1030,52.75,0.717857


In [ ]:
print(len(df))

4000


In [ ]:
flagged_check.sort_values("avg_attendance", ascending=True)

,Student ID,Semester 1 average,avg_attendance
336,337,52.21,0.714286
1029,1030,52.75,0.717857
2669,2670,51.96,0.732143
3961,3962,51.65,0.760714
2890,2891,51.82,0.764286
...,...,...,...
217,218,52.07,0.953571
3570,3571,52.05,0.957143
2271,2272,48.10,0.967857
2798,2799,52.72,1.000000


In [ ]:
def split_class_level(df):
    df = df.copy()

    def parse(val):
        if pd.isna(val):
            return (np.nan, np.nan, np.nan)
        s = str(val).strip()
        m = re.match(r"^(\d+)([A-Za-z])?(\d+)?$", s)
        if not m:
            return (np.nan, np.nan, np.nan)
        grade = int(m.group(1))
        stream = m.group(2) if m.group(2) else np.nan
        subgroup = int(m.group(3)) if m.group(3) else np.nan
        return (grade, stream, subgroup)

    parsed = df["Class level"].apply(parse)
    df["grade_number"] = parsed.apply(lambda t: t[0])
    df["stream_letter"] = parsed.apply(lambda t: t[1])
    df["subgroup"] = parsed.apply(lambda t: t[2])
    return df

df = split_class_level(df)
df[["Student ID", "Class level", "grade_number", "stream_letter", "subgroup"]]

,Student ID,Class level,grade_number,stream_letter,subgroup
0,1,4A,4.0,A,NaN
1,2,4B,4.0,B,NaN
2,3,7B3,7.0,B,3.0
3,4,5,5.0,NaN,NaN
4,5,6B,6.0,B,NaN
...,...,...,...,...,...
3995,3996,JHS 2,NaN,NaN,NaN
3996,3997,6A,6.0,A,NaN
3997,3998,4,4.0,NaN,NaN
3998,3999,6,6.0,NaN,NaN


In [ ]:
df["Class level"].isna().sum()

np.int64(17)

In [ ]:
print(df.shape)
df["Class level"].isna().sum()

(4000, 37)


np.int64(17)

In [ ]:
missing_raw = df["Class level"].isna()
unparsed = df["Class level"].notna() & df["grade_number"].isna()

print(f"Originally missing: {missing_raw.sum()}")
print(f"Non-null but failed to parse: {unparsed.sum()}")
df.loc[unparsed, "Class level"].unique()

Originally missing: 17
Non-null but failed to parse: 895


array(['JHS 1', 'JHS 1B1', 'JHS 2', 'JHS 1B2'], dtype=object)

In [ ]:
def split_class_level(df):
    df = df.copy()

    def parse(val):
        if pd.isna(val):
            return (np.nan, np.nan, np.nan, np.nan)
        s = str(val).strip()
        m = re.match(r"^(?:([A-Za-z]+)\s*)?(\d+)([A-Za-z])?(\d+)?$", s)
        if not m:
            return (np.nan, np.nan, np.nan, np.nan)
        prefix = m.group(1) if m.group(1) else np.nan
        grade = int(m.group(2))
        stream = m.group(3) if m.group(3) else np.nan
        subgroup = int(m.group(4)) if m.group(4) else np.nan
        return (prefix, grade, stream, subgroup)

    parsed = df["Class level"].apply(parse)
    df["level_prefix"] = parsed.apply(lambda t: t[0])
    df["grade_number"] = parsed.apply(lambda t: t[1])
    df["stream_letter"] = parsed.apply(lambda t: t[2])
    df["subgroup"] = parsed.apply(lambda t: t[3])
    return df

df = split_class_level(df)

missing_raw = df["Class level"].isna()
unparsed = df["Class level"].notna() & df["grade_number"].isna()
print(f"Originally missing: {missing_raw.sum()}")
print(f"Non-null but failed to parse: {unparsed.sum()}")
df.loc[unparsed, "Class level"].unique()

Originally missing: 17
Non-null but failed to parse: 0


array([], dtype=object)

In [ ]:
def standardize_categoricals(df):
    df = df.copy()
    text_cols = [
        "Child labor involvement",
        "Extra-curricular activities",
        "Family dropout history",
        "Mode of transport",
        "Teacher relationship quality",
        "Peer relationship quality",
        "Gender",
    ]
    for col in text_cols:
        if col in df.columns:
            df[col] = df[col].astype(str).str.strip()
    return df

def drop_identifiers(df):
    df = df.copy()
    if "Name" in df.columns:
        df = df.drop(columns=["Name"])
    return df

df = standardize_categoricals(df)
df = drop_identifiers(df)

print("Name" in df.columns)  # should print False
df["Child labor involvement"].value_counts()

False


,count
Child labor involvement,
No,1739
Yes,1369
Sometimes,892


In [ ]:
for col in ["Family dropout history", "Child labor involvement",
            "Teacher relationship quality", "Mode of transport",
            "Extra-curricular activities"]:
    print(f"--- {col} ---")
    print(df[col].unique())
    print()

--- Family dropout history ---
['No' 'Yes' 'Not sure']

--- Child labor involvement ---
['No' 'Yes' 'Sometimes']

--- Teacher relationship quality ---
['Poor' 'Good' 'Average' 'Very Good']

--- Mode of transport ---
['Walking' 'Bus' 'Public transport' 'Car']

--- Extra-curricular activities ---
['Football' 'Drawing' 'Watch cartoons' 'Dancing' 'Quiz' 'Quiz Club'
 'Sport Club' 'Drama' 'Sports' 'Sport' 'Reading' 'Singing' 'Dance' 'TV'
 'Watch tiktoks' 'Debate Club' 'Main Sanitation' 'Learning' 'Dance Club'
 'Video Games' 'Writing' 'nan' 'Tennis' 'Sweeping']



In [ ]:
def standardize_categoricals_v2(df):
    df = df.copy()

    transport_map = {
        "Walking": "Walking",
        "Bus": "Public transport",
        "Public transport": "Public transport",
        "Car": "Private transport",
    }
    df["Mode of transport"] = df["Mode of transport"].map(transport_map).fillna(df["Mode of transport"])

    activity_map = {
        "Football": "Sports", "Sports": "Sports", "Sport": "Sports", "Tennis": "Sports",
        "Drawing": "Arts & Performance", "Dancing": "Arts & Performance",
        "Dance": "Arts & Performance", "Dance Club": "Arts & Performance",
        "Singing": "Arts & Performance", "Drama": "Arts & Performance",
        "Quiz": "Academic/Clubs", "Quiz Club": "Academic/Clubs",
        "Debate Club": "Academic/Clubs", "Reading": "Academic/Clubs",
        "Writing": "Academic/Clubs", "Learning": "Academic/Clubs",
        "Watch cartoons": "Passive screen time", "Watch tiktoks": "Passive screen time",
        "TV": "Passive screen time", "Video Games": "Passive screen time",
        "Main Sanitation": "Chores/Duties", "Sweeping": "Chores/Duties",
    }
    df["Extra-curricular activities"] = df["Extra-curricular activities"].map(activity_map).fillna("None")

    return df

df = standardize_categoricals_v2(df)

print(df["Mode of transport"].value_counts())
print()
print(df["Extra-curricular activities"].value_counts())

Mode of transport
Public transport     1747
Walking              1564
Private transport     689
Name: count, dtype: int64

Extra-curricular activities
Arts & Performance     1892
Academic/Clubs          839
Sports                  802
None                    226
Passive screen time     190
Chores/Duties            51
Name: count, dtype: int64


In [ ]:
for col in ["Gender", "Peer relationship quality", "Section", "School", "Parental educational level"]:
    print(f"--- {col} ---")
    print(df[col].unique())
    print()

# check whether standardized income still lines up with the now-clean raw income
print(df[["Household income level", "Household income level (standardized)"]].drop_duplicates().sort_values("Household income level"))

--- Gender ---
['Female' 'Male']

--- Peer relationship quality ---
['Very Good' 'Average' 'Good' 'Poor']

--- Section ---
['Kofi Annan' 'K.A. Busia' 'Kwegyir Aggrey' 'Kwame Nkrumah' 'James Aggrey']

--- School ---
['Shining Star Preparatory' 'Weweso MA' 'Ayeduase RC']

--- Parental educational level ---
['SHS' 'Tertiary' 'JHS' 'Basic' 'University' nan]

     Household income level Household income level (standardized)
10                      1.0                                   Low
4                       2.0                               Average
0                       3.0                                  High
2                       NaN                                   NaN
138                     NaN                                   Low
147                     NaN                               Average
254                     NaN                                  High


In [ ]:
def fix_parental_education(df, basic_means_jhs=True):
    df = df.copy()
    mapping = {"University": "Tertiary"}
    if basic_means_jhs:
        mapping["Basic"] = "JHS"
    df["Parental educational level"] = df["Parental educational level"].replace(mapping)
    return df

def reconcile_household_income(df):
    df = df.copy()
    label_to_num = {"Low": 1, "Average": 2, "High": 3}
    num_to_label = {v: k for k, v in label_to_num.items()}

    # fill raw numeric from standardized where raw is missing
    needs_raw = df["Household income level"].isna() & df["Household income level (standardized)"].notna()
    df.loc[needs_raw, "Household income level"] = df.loc[needs_raw, "Household income level (standardized)"].map(label_to_num)

    # fill standardized from raw where standardized is missing
    needs_label = df["Household income level (standardized)"].isna() & df["Household income level"].notna()
    df.loc[needs_label, "Household income level (standardized)"] = df.loc[needs_label, "Household income level"].map(num_to_label)

    still_missing = df["Household income level"].isna() & df["Household income level (standardized)"].isna()
    print(f"Rows with BOTH fields missing (unrecoverable): {still_missing.sum()}")
    return df

df = fix_parental_education(df)
df = reconcile_household_income(df)

print(df["Parental educational level"].unique())
print(df[["Household income level", "Household income level (standardized)"]].drop_duplicates().sort_values("Household income level"))

Rows with BOTH fields missing (unrecoverable): 262
['SHS' 'Tertiary' 'JHS' nan]
    Household income level Household income level (standardized)
10                     1.0                                   Low
4                      2.0                               Average
0                      3.0                                  High
2                      NaN                                   NaN


In [ ]:
from google.colab import files

df.to_excel("EarlyFlag_cleaned.xlsx", index=False)
files.download("EarlyFlag_cleaned.xlsx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import pandas as pd

df = pd.read_excel("/content/EarlyFlag_cleaned.xlsx")  # match your actual filename

def merge_good_ratings(df, canonical="Good"):
    df = df.copy()
    for col in ["Teacher relationship quality", "Peer relationship quality"]:
        if col in df.columns:
            df[col] = df[col].replace({"Good": canonical, "Very Good": canonical})
    return df

df = merge_good_ratings(df)

print(df["Teacher relationship quality"].value_counts())
print(df["Peer relationship quality"].value_counts())

Teacher relationship quality
Good       2261
Average    1015
Poor        724
Name: count, dtype: int64
Peer relationship quality
Good       2352
Average     877
Poor        771
Name: count, dtype: int64


In [ ]:
from google.colab import files

df.to_excel("EarlyFlag_cleaned.xlsx", index=False)
files.download("EarlyFlag_cleaned.xlsx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
df["income_missing"] = df["Household income level"].isna()
df["semester2_missing"] = df["Semester 2 average"].isna()
df["class_level_missing"] = df["Class level"].isna()

print(df["income_missing"].sum(), df["semester2_missing"].sum(), df["class_level_missing"].sum())

262 65 17


In [ ]:
baseline = df.loc[df["income_missing"] == False, "Household income level (standardized)"].value_counts(normalize=True)
print(baseline)

Household income level (standardized)
Average    0.454254
High       0.415195
Low        0.130551
Name: proportion, dtype: float64


In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import RandomForestClassifier

predictor_cols = ["School", "Mode of transport", "Travel time to school (minutes)", "Parental educational level"]

known = df[df["income_missing"] == False]
X_known = pd.get_dummies(known[predictor_cols], dummy_na=True)
y_known = known["Household income level (standardized)"]

clf = RandomForestClassifier(n_estimators=200, random_state=42, class_weight="balanced")
scores = cross_val_score(clf, X_known, y_known, cv=5)
print("Cross-validated accuracy:", scores.mean())
print("Per-fold:", scores)

Cross-validated accuracy: 0.3485900822541503
Per-fold: [0.32219251 0.34759358 0.34625668 0.35609103 0.3708166 ]


In [ ]:
still_missing = df["Household income level"].isna() & df["Household income level (standardized)"].isna()
print(f"Rows with BOTH fields missing (unrecoverable): {still_missing.sum()}")

Rows with BOTH fields missing (unrecoverable): 262


In [ ]:
!pip install ctgan

from synthesize_missing_values import run_synthetic_fill
import pandas as pd

df = pd.read_excel("/content/EarlyFlag_cleaned.xlsx")
filled_df = run_synthetic_fill(df, epochs=300)

filled_df.to_excel("EarlyFlag_cleaned_synthfilled.xlsx", index=False)

from google.colab import files
files.download("EarlyFlag_cleaned_synthfilled.xlsx")

Training CTGAN on 3738 known rows for ['Household income level (standardized)'] (context: ['School', 'Mode of transport', 'Parental educational level'])...


Gen. (-02.16) | Discrim. (-00.01): 100%|██████████| 300/300 [02:23<00:00,  2.09it/s]


Filled 262 row(s) for ['Household income level (standardized)'].
Training CTGAN on 3935 known rows for ['Semester 2 average'] (context: ['Semester 1 average'])...


Gen. (-01.29) | Discrim. (-00.04): 100%|██████████| 300/300 [01:50<00:00,  2.70it/s]


Filled 65 row(s) for ['Semester 2 average'].
Training CTGAN on 2496 known rows for ['grade_number', 'stream_letter'] (context: ['School'])...


Gen. (-01.21) | Discrim. (-00.08): 100%|██████████| 300/300 [01:20<00:00,  3.73it/s]


Filled 17 row(s) for ['grade_number', 'stream_letter'].


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import pandas as pd

df_original = pd.read_excel("/content/EarlyFlag_cleaned.xlsx")           # fallback, NaN-flagged
df_filled = pd.read_excel("/content/EarlyFlag_cleaned_synthfilled.xlsx") # new, synthetic-filled

In [ ]:
print(df_original.shape, df_filled.shape)

(4000, 37) (4000, 37)


In [ ]:
targets = ["Household income level", "Household income level (standardized)",
           "Semester 2 average", "grade_number", "stream_letter"]
print(df_filled[targets].isna().sum())

Household income level                      0
Household income level (standardized)       0
Semester 2 average                          0
grade_number                                0
stream_letter                            1487
dtype: int64


In [ ]:
print("Income (standardized) — original known rows only:")
print(df_original["Household income level (standardized)"].value_counts(normalize=True))
print()
print("Income (standardized) — filled file, all rows:")
print(df_filled["Household income level (standardized)"].value_counts(normalize=True))

Income (standardized) — original known rows only:
Household income level (standardized)
Average    0.454254
High       0.415195
Low        0.130551
Name: proportion, dtype: float64

Income (standardized) — filled file, all rows:
Household income level (standardized)
Average    0.45625
High       0.40925
Low        0.13450
Name: proportion, dtype: float64


In [ ]:
print(df_original["Semester 2 average"].describe())
print(df_filled["Semester 2 average"].describe())

count    3935.000000
mean       41.162752
std        35.712152
min         0.000000
25%         5.320000
50%        49.480000
75%        74.840000
max       100.000000
Name: Semester 2 average, dtype: float64
count    4000.000000
mean       41.235626
std        35.681295
min        -0.422870
25%         5.340000
50%        49.615000
75%        74.780000
max       100.744057
Name: Semester 2 average, dtype: float64


In [ ]:
label_to_num = {"Low": 1, "Average": 2, "High": 3}
mismatch = df_filled[df_filled["Household income level"] != df_filled["Household income level (standardized)"].map(label_to_num)]
print(len(mismatch))

0


In [ ]:
print(df_filled["Semester 2 average"].min(), df_filled["Semester 2 average"].max())

-0.4228699516067067 100.74405702055


In [ ]:
untouched_cols = [c for c in df_original.columns if c not in targets]
comparison = (df_original[untouched_cols].fillna("NA_MARKER") == df_filled[untouched_cols].fillna("NA_MARKER"))
print(comparison.all().value_counts())

True     31
False     1
Name: count, dtype: int64


In [ ]:
filled_income_rows = df_original[df_original["Household income level"].isna()].index[:5]
print(df_filled.loc[filled_income_rows, ["Student ID", "School", "Mode of transport",
                                          "Parental educational level",
                                          "Household income level (standardized)"]])

    Student ID                    School Mode of transport  \
2            3  Shining Star Preparatory           Walking   
32          33               Ayeduase RC           Walking   
71          72  Shining Star Preparatory  Public transport   
86          87  Shining Star Preparatory  Public transport   
99         100                 Weweso MA           Walking   

   Parental educational level Household income level (standardized)  
2                    Tertiary                               Average  
32                        JHS                                   Low  
71                        NaN                               Average  
86                        SHS                                  High  
99                   Tertiary                               Average  


In [ ]:
# find the mystery mismatched column from check 6
untouched_cols = [c for c in df_original.columns if c not in
                  ["Household income level", "Household income level (standardized)",
                   "Semester 2 average", "grade_number", "stream_letter"]]
comparison = (df_original[untouched_cols].fillna("NA_MARKER") == df_filled[untouched_cols].fillna("NA_MARKER"))
bad_col = comparison.all()[comparison.all() == False].index[0]
print("Mismatched column:", bad_col)

diff_rows = df_original[df_original[bad_col].fillna("NA_MARKER") != df_filled[bad_col].fillna("NA_MARKER")]
print(diff_rows[[bad_col]].head(10))
print(df_filled.loc[diff_rows.index, [bad_col]].head(10))

Mismatched column: Semester difference
     Semester difference
5                    NaN
17                   NaN
21                   NaN
200                  NaN
217                  NaN
276                  NaN
328                  NaN
336                  NaN
449                  NaN
504                  NaN
     Semester difference
5              11.161894
17             -2.871868
21             -0.628970
200           -28.425894
217           -48.811260
276            34.445451
328           -47.099194
336           -42.735494
449            12.099350
504            10.722598


In [ ]:
# clip the out-of-range Semester 2 average values
out_of_range = (df_filled["Semester 2 average"] < 0) | (df_filled["Semester 2 average"] > 100)
print(f"Rows out of valid range: {out_of_range.sum()}")

df_filled["Semester 2 average"] = df_filled["Semester 2 average"].clip(0, 100)
df_filled["Semester difference"] = df_filled["Semester 2 average"] - df_filled["Semester 1 average"]

print(df_filled["Semester 2 average"].describe())

Rows out of valid range: 2
count    4000.000000
mean       41.235545
std        35.680864
min         0.000000
25%         5.340000
50%        49.615000
75%        74.780000
max       100.000000
Name: Semester 2 average, dtype: float64


In [ ]:
# Full-dataset check across every column
null_counts = df_filled.isna().sum()
remaining = null_counts[null_counts > 0]

if len(remaining) == 0:
    print("Clean — zero empty cells anywhere in the dataset.")
else:
    print("Still missing values in these columns:")
    print(remaining)

Still missing values in these columns:
Class level                          17
Parental educational level          208
Extra-curricular activities         226
Travel time to school (minutes)       8
stream_letter                      1487
subgroup                           2804
level_prefix                       3105
dtype: int64


In [ ]:
for col in remaining.index:
    if col == "stream_letter":
        continue
    print(f"\n--- {col}: {df_filled[col].isna().sum()} missing ---")
    print(df_filled[df_filled[col].isna()].head(5))


--- Class level: 17 missing ---
     Student ID                    School  Gender Class level  \
85           86  Shining Star Preparatory    Male         NaN   
221         222  Shining Star Preparatory    Male         NaN   
224         225  Shining Star Preparatory  Female         NaN   
238         239               Ayeduase RC  Female         NaN   
722         723                 Weweso MA    Male         NaN   

    Parental educational level  Week1_attendance  Week2_attendance  \
85                         SHS               1.0               0.2   
221                        SHS               1.0               0.5   
224                   Tertiary               1.0               1.0   
238                   Tertiary               1.0               1.0   
722                        SHS               1.0               1.0   

     Week3_attendance  Week4_attendance  Week5_attendance  ...  \
85                1.0               1.0               1.0  ...   
221               1.0  

In [ ]:
import os
print(os.listdir("/content"))

['.config', 'synthesize_missing_values.py', 'EarlyFlag_cleaned.xlsx', '__pycache__', '.ipynb_checkpoints', 'EarlyFlag_cleaned_synthfilled.xlsx', 'sample_data']


In [ ]:
filled_df = df_filled  # if not already the same variable
filled_df.to_excel("EarlyFlag_cleaned_synthfilled_final.xlsx", index=False)

from google.colab import files
files.download("EarlyFlag_cleaned_synthfilled_final.xlsx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# Full-dataset check across every column
null_counts = df_filled.isna().sum()
remaining = null_counts[null_counts > 0]

if len(remaining) == 0:
    print("Clean — zero empty cells anywhere in the dataset.")
else:
    print("Still missing values in these columns:")
    print(remaining)

Still missing values in these columns:
Class level                          17
Parental educational level          208
Extra-curricular activities         226
Travel time to school (minutes)       8
stream_letter                      1487
subgroup                           2804
level_prefix                       3105
dtype: int64


In [ ]:
import pandas as pd

df_final = pd.read_excel("/content/EarlyFlag_cleaned_synthfilled_final.xlsx")

null_counts = df_final.isna().sum()
remaining = null_counts[null_counts > 0]

if len(remaining) == 0:
    print("Clean — zero empty cells anywhere in the dataset.")
else:
    print("Still missing values in these columns:")
    print(remaining)

Still missing values in these columns:
Class level                          17
Parental educational level          208
Extra-curricular activities         226
Travel time to school (minutes)       8
stream_letter                      1487
subgroup                           2804
level_prefix                       3105
dtype: int64


In [ ]:
print(df_final["Extra-curricular activities"].unique())

['Sports' 'Arts & Performance' 'Passive screen time' 'Academic/Clubs' nan
 'Chores/Duties']


In [ ]:
df_final["Extra-curricular activities"] = df_final["Extra-curricular activities"].fillna("None")
print(df_final["Extra-curricular activities"].isna().sum())  # should be 0

0


In [ ]:
missing_travel = df_final[df_final["Travel time to school (minutes)"].isna()]
print(missing_travel["Travel time to school"].unique())

['3o minutes' 'car']


In [ ]:
from synthesize_missing_values import patch_remaining_gaps
import pandas as pd

df_final = pd.read_excel("/content/EarlyFlag_cleaned_synthfilled_final.xlsx")
df_final = patch_remaining_gaps(df_final, epochs=300)

# full re-check across every column
remaining = df_final.isna().sum()
remaining = remaining[remaining > 0]
print(remaining)

ImportError: cannot import name 'patch_remaining_gaps' from 'synthesize_missing_values' (/content/synthesize_missing_values.py)

In [ ]:
!cat /content/synthesize_missing_values.py | grep "def patch_remaining_gaps"

def patch_remaining_gaps(df, epochs=300):


In [ ]:
import importlib
import synthesize_missing_values
importlib.reload(synthesize_missing_values)
from synthesize_missing_values import patch_remaining_gaps

In [ ]:
import pandas as pd

df_final = pd.read_excel("/content/EarlyFlag_cleaned_synthfilled_final.xlsx")
df_final = patch_remaining_gaps(df_final, epochs=300)

remaining = df_final.isna().sum()
remaining = remaining[remaining > 0]
print(remaining)

NameError: name 're' is not defined

In [ ]:
import importlib
import synthesize_missing_values
importlib.reload(synthesize_missing_values)
from synthesize_missing_values import patch_remaining_gaps

In [ ]:
import pandas as pd

df_final = pd.read_excel("/content/EarlyFlag_cleaned_synthfilled_final.xlsx")
df_final = patch_remaining_gaps(df_final, epochs=300)

remaining = df_final.isna().sum()
remaining = remaining[remaining > 0]
print(remaining)

[travel_time_typo_fix] recovered 7 row(s) via typo correction
Training CTGAN on 3999 known rows for ['Travel time to school (minutes)'] (context: ['Mode of transport', 'School'])...


Gen. (-01.44) | Discrim. (-00.07): 100%|██████████| 300/300 [02:39<00:00,  1.88it/s]


Filled 1 row(s) for ['Travel time to school (minutes)'].
Training CTGAN on 3792 known rows for ['Parental educational level'] (context: ['School', 'Household income level (standardized)', 'Mode of transport'])...


Gen. (-01.88) | Discrim. (-00.01): 100%|██████████| 300/300 [02:36<00:00,  1.92it/s]

Filled 208 row(s) for ['Parental educational level'].
Class level        17
stream_letter    1487
subgroup         2804
level_prefix     3105
dtype: int64


In [ ]:
print(df_final["Parental educational level"].value_counts(normalize=True))
print(df_final["Travel time to school (minutes)"].describe())

Parental educational level
Tertiary    0.45975
SHS         0.33225
JHS         0.20800
Name: proportion, dtype: float64
count    4000.000000
mean       31.702270
std        31.735852
min         1.000000
25%        10.000000
50%        30.000000
75%        45.000000
max       300.000000
Name: Travel time to school (minutes), dtype: float64


In [ ]:
null_counts = df_final.isna().sum()
remaining = null_counts[null_counts > 0]

if len(remaining) == 0:
    print("Clean — zero empty cells anywhere in the dataset.")
else:
    print("Still missing values in these columns:")
    print(remaining)

Still missing values in these columns:
Class level        17
stream_letter    1487
subgroup         2804
level_prefix     3105
dtype: int64


In [ ]:
print(df_final[df_final["Travel time to school (minutes)"] > 120][["Student ID", "School", "Mode of transport", "Travel time to school (minutes)"]])

      Student ID                    School  Mode of transport  \
128          129  Shining Star Preparatory            Walking   
143          144               Ayeduase RC            Walking   
532          533                 Weweso MA   Public transport   
647          648  Shining Star Preparatory            Walking   
695          696                 Weweso MA   Public transport   
885          886  Shining Star Preparatory            Walking   
907          908  Shining Star Preparatory  Private transport   
910          911                 Weweso MA            Walking   
948          949                 Weweso MA  Private transport   
1097        1098  Shining Star Preparatory  Private transport   
1317        1318  Shining Star Preparatory            Walking   
1339        1340               Ayeduase RC  Private transport   
1672        1673               Ayeduase RC   Public transport   
1692        1693  Shining Star Preparatory   Public transport   
1784        1785  Shining

In [ ]:
n_over_2hr = (df_final["Travel time to school (minutes)"] > 120).sum()
print(f"Rows with travel time > 2 hours: {n_over_2hr}")
print(df_final[df_final["Travel time to school (minutes)"] > 120][
    ["Student ID", "School", "Mode of transport", "Travel time to school (minutes)"]
])

Rows with travel time > 2 hours: 32
      Student ID                    School  Mode of transport  \
128          129  Shining Star Preparatory            Walking   
143          144               Ayeduase RC            Walking   
532          533                 Weweso MA   Public transport   
647          648  Shining Star Preparatory            Walking   
695          696                 Weweso MA   Public transport   
885          886  Shining Star Preparatory            Walking   
907          908  Shining Star Preparatory  Private transport   
910          911                 Weweso MA            Walking   
948          949                 Weweso MA  Private transport   
1097        1098  Shining Star Preparatory  Private transport   
1317        1318  Shining Star Preparatory            Walking   
1339        1340               Ayeduase RC  Private transport   
1672        1673               Ayeduase RC   Public transport   
1692        1693  Shining Star Preparatory   Public tr

In [ ]:
df_final["long_commute_flag"] = (df_final["Travel time to school (minutes)"] > 120).astype(int)
df_final["long_walk_flag"] = (
    (df_final["Travel time to school (minutes)"] > 120) &
    (df_final["Mode of transport"] == "Walking")
).astype(int)

print(df_final["long_commute_flag"].value_counts())
print(df_final["long_walk_flag"].value_counts())

long_commute_flag
0    3968
1      32
Name: count, dtype: int64
long_walk_flag
0    3982
1      18
Name: count, dtype: int64


In [ ]:
df_final.to_excel("EarlyFlag_cleaned_FINAL_v2.xlsx", index=False)
from google.colab import files
files.download("EarlyFlag_cleaned_FINAL_v2.xlsx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
null_counts = df_final.isna().sum()
remaining = null_counts[null_counts > 0]

if len(remaining) == 0:
    print("Clean — zero empty cells anywhere in the dataset.")
else:
    print("Still missing values in these columns:")
    print(remaining)

Still missing values in these columns:
Class level        17
stream_letter    1487
subgroup         2804
level_prefix     3105
dtype: int64
